In [13]:
import pandas as pd

In [14]:
# Replace this filename with the exact Vahan CSV filename in data/raw/
vahan_path = "../data/raw/vahan_vehicle_registrations_raw.csv"

vahan_df = pd.read_csv(vahan_path)

print("Vahan dataset loaded successfully.")
print(f"Shape: {vahan_df.shape}")

Vahan dataset loaded successfully.
Shape: (213405, 5)


In [15]:
print("First 5 rows:")
display(vahan_df.head())

print("\nColumn names:")
print(vahan_df.columns.tolist())

print("\nData types:")
display(vahan_df.dtypes)

print("\nDataset information:")
vahan_df.info()

First 5 rows:


,id,state,rto,maker,count
0,2205,Andhra Pradesh,Dharamavaram unit office,VE COMMERCIAL VEHICLES LTD,5
1,53310,Haryana,SDM KALANWALI,NEW GURDIAL AGRO INDUSTRIES PVT LTD,3
2,84859,Kerala,IRITTY SRTO,IDEAL JAWA INDIA PVT LTD,3
3,140878,Rajasthan,JODHPUR RTO,EYAMAUTO MOBILITY PVT LTD,3
4,188823,Uttar Pradesh,BASTI RTO,TEREX CORPORATION,3



Column names:
['id', 'state', 'rto', 'maker', 'count']

Data types:


id       int64
state      str
rto        str
maker      str
count    int64
dtype: object


Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 213405 entries, 0 to 213404
Data columns (total 5 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   id      213405 non-null  int64
 1   state   213405 non-null  str  
 2   rto     213405 non-null  str  
 3   maker   213405 non-null  str  
 4   count   213405 non-null  int64
dtypes: int64(2), str(3)
memory usage: 8.1 MB


## Initial Assessment of the Vahan Dataset

The Vahan dataset contains 213,405 records with information on State/UT,
RTO, vehicle maker and vehicle count.

Each record represents a vehicle-count entry associated with a specific
State/UT, RTO and maker.

The dataset does not contain an explicit year field in the downloaded
resource. Therefore, the temporal definition of the `count` variable must
be established from the source metadata before using the dataset as a
denominator for accident-rate calculations.

The dataset will therefore not yet be used to calculate State/UT accident
rates. Further validation of its reference period and measurement definition
is required.

## 2. 2024 State/UT Population Context

This section creates a State/UT-level population table from the official
Government of India population projections.

Source:
National Commission on Population, Ministry of Health and Family Welfare,
Report of the Technical Group on Population Projections, November 2019.

The source table is Table 1.1.3, "State/UT wise distribution of Projected
Population of India, 2011–2036."

The source reports population figures in thousands ('000). Therefore, the
2024 values are multiplied by 1,000 to obtain population counts in persons.

These are projected population values, not a 2024 Census enumeration.

In [16]:
# RoadSafe India — 2024 State/UT Population Dataset

population_data = {
    "state": [
        "Andhra Pradesh",
        "Arunachal Pradesh",
        "Assam",
        "Bihar",
        "Chhattisgarh",
        "Goa",
        "Gujarat",
        "Haryana",
        "Himachal Pradesh",
        "Jammu & Kashmir",
        "Jharkhand",
        "Karnataka",
        "Kerala",
        "Madhya Pradesh",
        "Maharashtra",
        "Manipur",
        "Meghalaya",
        "Mizoram",
        "Nagaland",
        "Odisha",
        "Punjab",
        "Rajasthan",
        "Sikkim",
        "Tamil Nadu",
        "Telangana",
        "Tripura",
        "Uttar Pradesh",
        "Uttarakhand",
        "West Bengal",
        "Andaman & Nicobar Islands",
        "Chandigarh",
        "Dadra & Nagar Haveli",
        "Daman & Diu",
        "N.C.T of Delhi",
        "Lakshadweep",
        "Puducherry",
        "Ladakh"
    ],
    
    # Source values are in thousands ('000)
    "population_2024_thousands": [
        53340,
        1576,
        36047,
        128592,
        30524,
        1583,
        72367,
        30573,
        7505,
        13701,
        39963,
        68115,
        35920,
        87610,
        127360,
        3253,
        3379,
        1250,
        2253,
        44420,
        30926,
        81897,
        695,
        77089,
        38272,
        4184,
        238078,
        11755,
        99563,
        404,
        1243,
        745,
        611,
        21752,
        69,
        1683,
        302
    ]
}

population_df = pd.DataFrame(population_data)

# Convert thousands into actual population counts
population_df["population_2024"] = (
    population_df["population_2024_thousands"] * 1000
)

display(population_df)

,state,population_2024_thousands,population_2024
0,Andhra Pradesh,53340,53340000
1,Arunachal Pradesh,1576,1576000
2,Assam,36047,36047000
3,Bihar,128592,128592000
4,Chhattisgarh,30524,30524000
5,Goa,1583,1583000
6,Gujarat,72367,72367000
7,Haryana,30573,30573000
8,Himachal Pradesh,7505,7505000
9,Jammu & Kashmir,13701,13701000


In [17]:
# Basic validation

print(f"Number of State/UT records: {len(population_df)}")

print("\nMissing values:")
display(population_df.isnull().sum())

print("\nDuplicate State/UT names:")
display(
    population_df[
        population_df["state"].duplicated(keep=False)
    ]
)

print("\nPopulation summary:")
display(
    population_df["population_2024"].describe()
)

Number of State/UT records: 37

Missing values:


state                        0
population_2024_thousands    0
population_2024              0
dtype: int64


Duplicate State/UT names:


,state,population_2024_thousands,population_2024



Population summary:


count    3.700000e+01
mean     3.779997e+07
std      5.024675e+07
min      6.900000e+04
25%      1.583000e+06
50%      2.175200e+07
75%      5.334000e+07
max      2.380780e+08
Name: population_2024, dtype: float64

In [18]:
# Validate State/UT population total against the published India total

state_population_total = population_df["population_2024"].sum()
official_india_population_2024 = 1_398_598_000

difference = (
    state_population_total - official_india_population_2024
)

difference_percent = (
    difference / official_india_population_2024 * 100
)

print(f"Sum of State/UT population: {state_population_total:,.0f}")
print(f"Published India projection: {official_india_population_2024:,.0f}")
print(f"Difference: {difference:,.0f} persons")
print(f"Difference percentage: {difference_percent:.6f}%")

Sum of State/UT population: 1,398,599,000
Published India projection: 1,398,598,000
Difference: 1,000 persons
Difference percentage: 0.000072%


In [19]:
# Display the population table sorted by State/UT
# so we can compare the manually entered values with the source.

display(
    population_df[
        ["state", "population_2024_thousands"]
    ].sort_values("state")
)

,state,population_2024_thousands
29,Andaman & Nicobar Islands,404
0,Andhra Pradesh,53340
1,Arunachal Pradesh,1576
2,Assam,36047
3,Bihar,128592
30,Chandigarh,1243
4,Chhattisgarh,30524
31,Dadra & Nagar Haveli,745
32,Daman & Diu,611
5,Goa,1583


In [20]:
# Save the derived 2024 population dataset

population_output_path = (
    "../data/processed/state_population_2024.csv"
)

population_df.to_csv(
    population_output_path,
    index=False
)

print(
    "2024 State/UT population dataset saved to:"
)
print(population_output_path)

2024 State/UT population dataset saved to:
../data/processed/state_population_2024.csv


## Population Validation Conclusion

The 2024 State/UT population values were transcribed from the official
Government of India population-projection table.

The sum of the published State/UT values is 1,398,599 thousand, while the
published India total is 1,398,598 thousand. The resulting discrepancy is
1 thousand persons (1,000 people), or approximately 0.000072% of the
published national projection.

Because the individual State/UT values match the source table, the values
are retained without artificial adjustment. The discrepancy is documented
as a minor reconciliation difference in the source table.